# Compositionality Shared Task - ModernBERT Baseline

## Subtask A: Noun Compounds (English)

**Model:** ModernBERT-base (answerdotai/ModernBERT-base)

**Features:**
- [CLS] token embedding for Mod, Head, Compound, Context (768d each)
- Cosine similarity between Mod and Head embeddings
- Total: 768*4 + 1 = 3073 features

**Approach:**
1. Load data + create stratified folds
2. Encode with ModernBERT
3. Train Ridge regression for ModAvg and HeadAvg
4. Evaluate with Spearman ρ

In [ ]:
import os
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import mean_squared_error
import torch
from transformers import AutoTokenizer, AutoModel
import warnings
warnings.filterwarnings('ignore')

# Paths
KAGGLE_PATH = '/kaggle/input/datasets/ieltsmater/compartment'
OUTPUT_DIR = '/kaggle/working'

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Check GPU
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 1. Load Data & Create Stratified Folds

In [ ]:
def prepare_stratified_folds(df, target_col='Compound', n_splits=5, seed=42):
    """
    Create stratified folds for Subtask A (Noun Compounds).
    Uses average of ModAvg and HeadAvg for binning.
    """
    df = df.copy()
    
    # 1. Compute mean compositionality per target
    df['mean_score'] = (df['ModAvg'] + df['HeadAvg']) / 2
    target_stats = df.groupby(target_col)['mean_score'].mean().reset_index()
    
    # 2. Bin targets into 3 categories
    target_stats['bin'] = pd.qcut(
        target_stats['mean_score'], q=3, labels=[0, 1, 2], duplicates='drop'
    )
    
    # 3. Split on unique targets
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    
    target_folds = {}
    for fold, (_, val_idx) in enumerate(
        sgkf.split(target_stats[target_col], y=target_stats['bin'], groups=target_stats[target_col])
    ):
        for idx in val_idx:
            target_folds[target_stats[target_col].iloc[idx]] = fold
    
    # 4. Map folds back to all rows
    df['fold'] = df[target_col].map(target_folds)
    df.drop(columns=['mean_score'], inplace=True)
    
    return df

In [ ]:
# Load training data
df = pd.read_csv(f'{KAGGLE_PATH}/dataset/en-nn-train.tsv', sep='\t')
df = prepare_stratified_folds(df, target_col='Compound')

print(f'Dataset shape: {df.shape}')
print(f'Unique compounds: {df["Compound"].nunique()}')
print(f'\nRows per fold:')
print(df.groupby('fold').size())

## 2. Load ModernBERT

In [ ]:
MODEL_NAME = 'answerdotai/ModernBERT-base'

print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

print(f'Model loaded! Hidden size: {model.config.hidden_size}')

## 3. Encode with ModernBERT

In [ ]:
def encode_texts(texts, tokenizer, model, batch_size=32, max_length=128):
    """
    Encode list of texts using [CLS] token embedding.
    Returns numpy array of shape (n_texts, hidden_size).
    """
    all_embeddings = []
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        
        # Tokenize
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        ).to(device)
        
        # Get [CLS] embedding
        with torch.no_grad():
            outputs = model(**inputs)
            cls_embeddings = outputs.last_hidden_state[:, 0, :]  # [CLS] token
        
        all_embeddings.append(cls_embeddings.cpu().numpy())
        
        if (i // batch_size) % 10 == 0:
            print(f'  Encoded {min(i+batch_size, len(texts))}/{len(texts)}')
    
    return np.concatenate(all_embeddings, axis=0)

In [ ]:
# Encode all components
print('Encoding Mod...')
mod_embeddings = encode_texts(df['Mod'].astype(str).tolist(), tokenizer, model)

print('\nEncoding Head...')
head_embeddings = encode_texts(df['Head'].astype(str).tolist(), tokenizer, model)

print('\nEncoding Compound...')
compound_embeddings = encode_texts(df['Compound'].astype(str).tolist(), tokenizer, model)

print('\nEncoding Context...')
context_embeddings = encode_texts(df['Context'].astype(str).tolist(), tokenizer, model, max_length=512)

print(f'\nEmbedding shape: {mod_embeddings.shape}')

In [ ]:
# Build feature matrix
# Cosine similarity between mod and head
cos_sim = np.sum(mod_embeddings * head_embeddings, axis=1) / (
    np.linalg.norm(mod_embeddings, axis=1) * np.linalg.norm(head_embeddings, axis=1) + 1e-8
)

# Concatenate: cos_sim (1) + mod (768) + head (768) + compound (768) + context (768)
X = np.concatenate([
    cos_sim.reshape(-1, 1),
    mod_embeddings,
    head_embeddings,
    compound_embeddings,
    context_embeddings
], axis=1)

y_mod = df['ModAvg'].values
y_head = df['HeadAvg'].values
folds = df['fold'].values

print(f'X shape: {X.shape}')  # (n_samples, 3073)
print(f'y_mod shape: {y_mod.shape}')
print(f'y_head shape: {y_head.shape}')

## 4. 5-Fold Cross-Validation

In [ ]:
# OOF predictions
oof_mod = np.zeros(len(df))
oof_head = np.zeros(len(df))

results = []

for fold in range(5):
    print(f'\n--- Fold {fold} ---')
    
    # Split
    train_idx = folds != fold
    val_idx = folds == fold
    
    X_train, X_val = X[train_idx], X[val_idx]
    y_mod_train, y_mod_val = y_mod[train_idx], y_mod[val_idx]
    y_head_train, y_head_val = y_head[train_idx], y_head[val_idx]
    
    # Model for modifier
    pipe_mod = Pipeline([
        ('scaler', StandardScaler()),
        ('ridge', Ridge(alpha=1.0))
    ])
    pipe_mod.fit(X_train, y_mod_train)
    pred_mod = pipe_mod.predict(X_val)
    rho_mod = spearmanr(y_mod_val, pred_mod).statistic
    oof_mod[val_idx] = pred_mod
    
    # Model for head
    pipe_head = Pipeline([
        ('scaler', StandardScaler()),
        ('ridge', Ridge(alpha=1.0))
    ])
    pipe_head.fit(X_train, y_head_train)
    pred_head = pipe_head.predict(X_val)
    rho_head = spearmanr(y_head_val, pred_head).statistic
    oof_head[val_idx] = pred_head
    
    print(f'  Mod rho: {rho_mod:.4f}')
    print(f'  Head rho: {rho_head:.4f}')
    print(f'  Mean rho: {(rho_mod + rho_head) / 2:.4f}')
    
    results.append({
        'fold': fold,
        'rho_mod': rho_mod,
        'rho_head': rho_head,
        'rho_mean': (rho_mod + rho_head) / 2
    })

## 5. Results Summary

In [ ]:
results_df = pd.DataFrame(results)
print('=== Per-Fold Results ===')
print(results_df.to_string(index=False))

print(f'\n=== Overall ===')
print(f'Mod rho (mean +/- std):  {results_df["rho_mod"].mean():.4f} +/- {results_df["rho_mod"].std():.4f}')
print(f'Head rho (mean +/- std): {results_df["rho_head"].mean():.4f} +/- {results_df["rho_head"].std():.4f}')
print(f'Mean rho (overall):      {(results_df["rho_mod"].mean() + results_df["rho_head"].mean()) / 2:.4f}')

# Overall OOF Spearman
oof_rho_mod = spearmanr(y_mod, oof_mod).statistic
oof_rho_head = spearmanr(y_head, oof_head).statistic
print(f'\nOOF Mod rho:  {oof_rho_mod:.4f}')
print(f'OOF Head rho: {oof_rho_head:.4f}')
print(f'OOF Mean rho: {(oof_rho_mod + oof_rho_head) / 2:.4f}')

# RMSE
rmse_mod = np.sqrt(mean_squared_error(y_mod, oof_mod))
rmse_head = np.sqrt(mean_squared_error(y_head, oof_head))
print(f'\nOOF RMSE Mod:  {rmse_mod:.4f}')
print(f'OOF RMSE Head: {rmse_head:.4f}')

## 6. Predict on Trial Data & Generate Submission

In [ ]:
# 1. Train final models on ALL training data
scaler_mod = StandardScaler()
X_scaled = scaler_mod.fit_transform(X)

final_mod = Ridge(alpha=1.0)
final_mod.fit(X_scaled, y_mod)

final_head = Ridge(alpha=1.0)
final_head.fit(X_scaled, y_head)

print('Final models trained on all training data.')

In [ ]:
# 2. Load and encode trial data
df_trial = pd.read_csv(f'{KAGGLE_PATH}/trial/en-nn-trial.tsv', sep='\t')
print(f'Trial data shape: {df_trial.shape}')

# Encode
trial_mod_emb = encode_texts(df_trial['Mod'].astype(str).tolist(), tokenizer, model)
trial_head_emb = encode_texts(df_trial['Head'].astype(str).tolist(), tokenizer, model)
trial_compound_emb = encode_texts(df_trial['Compound'].astype(str).tolist(), tokenizer, model)
trial_context_emb = encode_texts(df_trial['Context'].astype(str).tolist(), tokenizer, model, max_length=512)

# Features
trial_cos_sim = np.sum(trial_mod_emb * trial_head_emb, axis=1) / (
    np.linalg.norm(trial_mod_emb, axis=1) * np.linalg.norm(trial_head_emb, axis=1) + 1e-8
)

X_trial = np.concatenate([
    trial_cos_sim.reshape(-1, 1),
    trial_mod_emb,
    trial_head_emb,
    trial_compound_emb,
    trial_context_emb
], axis=1)
X_trial_scaled = scaler_mod.transform(X_trial)

# Predict
trial_pred_mod = final_mod.predict(X_trial_scaled)
trial_pred_head = final_head.predict(X_trial_scaled)

print(f'Trial predictions shape: {trial_pred_mod.shape}')

In [ ]:
# 3. Evaluate on trial (gold labels available)
trial_rho_mod = spearmanr(df_trial['ModAvg'], trial_pred_mod).statistic
trial_rho_head = spearmanr(df_trial['HeadAvg'], trial_pred_head).statistic

trial_rmse_mod = np.sqrt(mean_squared_error(df_trial['ModAvg'], trial_pred_mod))
trial_rmse_head = np.sqrt(mean_squared_error(df_trial['HeadAvg'], trial_pred_head))

print('=== Trial Set Results ===')
print(f'Mod  Spearman rho: {trial_rho_mod:.4f}  |  RMSE: {trial_rmse_mod:.4f}')
print(f'Head Spearman rho: {trial_rho_head:.4f}  |  RMSE: {trial_rmse_head:.4f}')
print(f'Mean Spearman rho: {(trial_rho_mod + trial_rho_head) / 2:.4f}')

In [ ]:
# 4. Save submission file (no header, tab-separated)
os.makedirs(f'{OUTPUT_DIR}/submission', exist_ok=True)

submission = pd.DataFrame({
    'ContextID': df_trial['ContextID'],
    'Modifier': trial_pred_mod,
    'Head': trial_pred_head
})

submission.to_csv(f'{OUTPUT_DIR}/submission/en-nn-trial-pred.tsv', sep='\t', index=False, header=False)
print(f'Saved: {OUTPUT_DIR}/submission/en-nn-trial-pred.tsv')
print(f'\nSubmission preview:')
print(submission.to_string(index=False))

## Comparison: Word2Vec vs ModernBERT

| Metric | Word2Vec | ModernBERT |
|--------|----------|------------|
| OOF Mod rho | 0.1458 | ? |
| OOF Head rho | 0.2468 | ? |
| OOF Mean rho | 0.1963 | ? |

**Expected improvement:** ModernBERT should capture context better, especially for:
- Ambiguous compounds (e.g., "night watch" in different contexts)
- Modifier compositionality (which was weaker with Word2Vec)